In [86]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor

### Importing Data

In [87]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

### Data Description

In [88]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [89]:
drop_cols = ["Id","MiscFeature", "PoolQC","FireplaceQu","MasVnrType","Alley","Fence"]
train_df.drop(drop_cols, axis =1, inplace = True)

In [90]:
test_df.drop(drop_cols, axis = 1, inplace = True)

In [91]:
y1 = train_df["SalePrice"]

In [92]:
X = train_df.drop("SalePrice", axis = 1)
X_test = test_df
print(X.info())
print(X_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 73 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   LotShape       1460 non-null   object 
 6   LandContour    1460 non-null   object 
 7   Utilities      1460 non-null   object 
 8   LotConfig      1460 non-null   object 
 9   LandSlope      1460 non-null   object 
 10  Neighborhood   1460 non-null   object 
 11  Condition1     1460 non-null   object 
 12  Condition2     1460 non-null   object 
 13  BldgType       1460 non-null   object 
 14  HouseStyle     1460 non-null   object 
 15  OverallQual    1460 non-null   int64  
 16  OverallCond    1460 non-null   int64  
 17  YearBuilt      1460 non-null   int64  
 18  YearRemo

In [93]:
X_train, X_valid, y_train, y_valid = train_test_split(X,y1, test_size =0.3)

In [94]:
cat_cols = X_train.select_dtypes(include = "object").columns
num_cols = X_train.select_dtypes(include = ["float","int"]).columns

In [95]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1022 entries, 741 to 856
Data columns (total 73 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1022 non-null   int64  
 1   MSZoning       1022 non-null   object 
 2   LotFrontage    845 non-null    float64
 3   LotArea        1022 non-null   int64  
 4   Street         1022 non-null   object 
 5   LotShape       1022 non-null   object 
 6   LandContour    1022 non-null   object 
 7   Utilities      1022 non-null   object 
 8   LotConfig      1022 non-null   object 
 9   LandSlope      1022 non-null   object 
 10  Neighborhood   1022 non-null   object 
 11  Condition1     1022 non-null   object 
 12  Condition2     1022 non-null   object 
 13  BldgType       1022 non-null   object 
 14  HouseStyle     1022 non-null   object 
 15  OverallQual    1022 non-null   int64  
 16  OverallCond    1022 non-null   int64  
 17  YearBuilt      1022 non-null   int64  
 18  YearRemodAdd

### Handling Null Values

In [96]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy = 'most_frequent')

In [97]:
X_train[cat_cols] = pd.DataFrame(cat_imputer.fit(X_train[cat_cols]).transform(X_train[cat_cols]), 
                                 columns = cat_cols,index=X_train.index)
X_valid[cat_cols] = pd.DataFrame(cat_imputer.transform(X_valid[cat_cols]), columns = cat_cols,index=X_valid.index)
X_test[cat_cols] = pd.DataFrame(cat_imputer.transform(X_test[cat_cols]), columns = cat_cols,index=X_test.index)

In [98]:
X_train[num_cols] = pd.DataFrame(num_imputer.fit(X_train[num_cols]).transform(X_train[num_cols]),
                                 columns = num_cols, index = X_train.index)
X_valid[num_cols] = pd.DataFrame(num_imputer.transform(X_valid[num_cols]), columns = num_cols, index = X_valid.index)
X_test[num_cols] = pd.DataFrame(num_imputer.transform(X_test[num_cols]), columns = num_cols, index = X_test.index)

In [99]:
X_train.isna().sum().sum()

0

### Standardizing Numeric Values

In [100]:
scaler = StandardScaler() 
X_train[num_cols] = pd.DataFrame(scaler.fit(X_train[num_cols]).transform(X_train[num_cols]),
                                  columns=num_cols, index=X_train.index)
X_valid[num_cols] = pd.DataFrame(scaler.transform(X_valid[num_cols]),
                                  columns=num_cols, index=X_valid.index)
X_test[num_cols] = pd.DataFrame(scaler.transform(X_test[num_cols]),
                                 columns=num_cols, index=X_test.index)

### Encoding Categorical data

In [101]:
high_card_cols = [c for c in cat_cols if X_train[c].nunique() >= 10]
low_card_cols  = [c for c in cat_cols if X_train[c].nunique() < 10]

In [107]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train[low_card_cols] = pd.DataFrame(oe.fit_transform(X_train[low_card_cols]),
                                  columns=low_card_cols, index=X_train.index)
X_valid[low_card_cols] = pd.DataFrame(oe.transform(X_valid[low_card_cols]),
                                  columns=low_card_cols, index=X_valid.index)
X_test[low_card_cols]  = pd.DataFrame(oe.transform(X_test[low_card_cols]),
                                  columns=low_card_cols, index=X_test.index)

In [106]:
from sklearn.preprocessing import TargetEncoder

te = TargetEncoder(target_type='continuous', smooth='auto')

X_train[high_card_cols] = pd.DataFrame(te.fit_transform(X_train[high_card_cols], y_train),
                                        columns=high_card_cols, index=X_train.index)
X_valid[high_card_cols] = pd.DataFrame(te.transform(X_valid[high_card_cols]),
                                        columns=high_card_cols, index=X_valid.index)
X_test[high_card_cols]  = pd.DataFrame(te.transform(X_test[high_card_cols]),
                                        columns=high_card_cols, index=X_test.index)

### Random Forest Regressor

In [108]:
regressor = RandomForestRegressor(n_estimators = 100)
regressor.fit(X_train,y_train)

RandomForestRegressor()

In [109]:
y_pred = regressor.predict(X_valid)

In [110]:
rmse = root_mean_squared_error(y_pred, y_valid)
rmse

23336.438924265567

### XGBoost

In [111]:
model = XGBRegressor(objective='reg:squarederror',
                         n_estimators=100,learning_rate = 0.05, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_valid)

In [112]:
rmse = root_mean_squared_error(y_pred,y_valid)
rmse

23663.8359375